In [1]:
### STEP 1: Environment Setup & Paths Initialization
# Purpose: Synchronize the project root and define absolute path targets for ingestion and export.
# Input: Working directory state.
# Output: Configured Path objects for MLOps tracking.
# Expected Side Effects: Minimal. Standard out prints to confirm synchronized path resolutions.

import sys
from pathlib import Path
import polars as pl

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

TRAIN_RAW_CLEAN = PROJECT_ROOT / "data" / "processed" / "train_raw_clean.parquet"
TEST_RAW_CLEAN  = PROJECT_ROOT / "data" / "processed" / "test_raw_clean.parquet"
TRAIN_FEATURES  = PROJECT_ROOT / "data" / "processed" / "train_features.parquet"
TEST_FEATURES   = PROJECT_ROOT / "data" / "processed" / "test_features.parquet"

print(f"[MLOps] Ingestion paths synchronized.\nTrain Src: {TRAIN_RAW_CLEAN.name}\nTest Src: {TEST_RAW_CLEAN.name}")

[MLOps] Ingestion paths synchronized.
Train Src: train_raw_clean.parquet
Test Src: test_raw_clean.parquet


In [2]:
### STEP 2: Lazy Ingestion Scan
# Purpose: Initialize computational graphs using zero-copy lazy evaluations rather than eagerly reading matrices to RAM.
# Input: Disk addresses for TRAIN_RAW_CLEAN, TEST_RAW_CLEAN parquet sources.
# Output: Unresolved pl.LazyFrame nodes (train_lf, test_lf) maintaining full 78-column base dimension.
# Expected Side Effects: None. Data is not materialized in memory at this stage.

train_lf = pl.scan_parquet(str(TRAIN_RAW_CLEAN))
test_lf  = pl.scan_parquet(str(TEST_RAW_CLEAN))
print("[Pipeline] Lazy computation graphs initiated. Baseline feature space completely intact.")

[Pipeline] Lazy computation graphs initiated. Baseline feature space completely intact.


In [3]:
# STEP 3: Isolated Training Metrics Derivation - Cell 3

# Purpose: Extract median gas and P1/P99 outlier bounds strictly on the training split to eradicate forward-horizon data leakage.
# Input: TRAIN_RAW_CLEAN (isolated), train_lf (pure LazyFrame graph metadata).
# Output: train_gas_median (float), outlier_bounds (nested dictionary of thresholds).
# Expected Side Effects: Zero RAM volume footprint. Column detection utilizes schema metadata, and bounds compute via single parallel pass.


from src.features import fit_outlier_bounds

# 1.
train_gas_median = pl.scan_parquet(str(TRAIN_RAW_CLEAN)).select(pl.col("avg_gas_paid_per_tx_eth").median()).collect().item()
print(f"[Data Contract] Frozen Train Gas Median Extracted: {train_gas_median}")

# 2.
schema = train_lf.collect_schema()
numeric_cols = [name for name, dtype in schema.items() if dtype.is_numeric() and name != "target"]

# 3. 
outlier_bounds = fit_outlier_bounds(train_lf, numeric_cols)

[Data Contract] Frozen Train Gas Median Extracted: 0.01090902541576037
[OUTLIER FIT] Computed P1/P99 bounds for 76 numeric columns (train only, single-pass)


In [4]:
### STEP 4: Quantitative Feature Engineering Manifold
# Purpose: Append 5 specialized DeFi credit risk metrics to both streaming splits.
# Input: train_lf, test_lf, train_gas_median.
# Output: train_lf, test_lf arrays fully expanded with high-dimensional predictive signals.
# Expected Side Effects: Lazily expands the execution graph schema to incorporate 5 custom domains.

def engineer_defi_signals(lf: pl.LazyFrame, gas_median_scalar: float) -> pl.LazyFrame:
    safe_gas_median = gas_median_scalar if (gas_median_scalar is not None and gas_median_scalar > 0) else 1e-6
    return lf.with_columns([
        pl.when(pl.col("total_collateral_eth") > 0)
        .then(pl.col("borrow_amount_sum_eth") / pl.col("total_collateral_eth"))
        .otherwise(pl.lit(None, dtype=pl.Float64)).alias("LTV_Utilization_Quotient"),
        
        pl.when(pl.col("borrow_amount_sum_eth") > 0)
        .then(pl.col("repay_amount_sum_eth") / pl.col("borrow_amount_sum_eth"))
        .otherwise(pl.lit(None, dtype=pl.Float64)).alias("Debt_Service_Coverage_Proxy"),
        
        (pl.col("total_collateral_eth") - pl.col("borrow_amount_sum_eth")).alias("Net_Unencumbered_Collateral_Base"),
        
        (pl.col("avg_gas_paid_per_tx_eth") / safe_gas_median).alias("Gas_Expenditure_Premium_Ratio"),
        
        pl.when(pl.col("wallet_age") > 0)
        .then((pl.col("incoming_tx_count") + pl.col("outgoing_tx_count")) / pl.col("wallet_age").log1p())
        .otherwise(pl.lit(None, dtype=pl.Float64)).alias("LogSmoothed_Network_Velocity_Index")
    ])

train_lf = engineer_defi_signals(train_lf, train_gas_median)
test_lf = engineer_defi_signals(test_lf, train_gas_median)
print("[MLOps] 5 Numerically Stable DeFi Credit Features appended to the analytical schema.")

[MLOps] 5 Numerically Stable DeFi Credit Features appended to the analytical schema.


In [5]:
### STEP 5: Defensive Outlier Clamping
# Purpose: Dynamically inject clip constraints onto the numerical streams using pre-calculated in-sample bounds.
# Input: Lazy streams train_lf, test_lf, and outlier_bounds dictionary.
# Output: Clamped train_lf, test_lf nodes immune to extreme distribution anomalies.
# Expected Side Effects: Updates the lazy computation graph expression tree with bounding scalar logic.

from src.features import apply_clamping

train_lf = apply_clamping(train_lf, outlier_bounds, split_tag="train")
test_lf = apply_clamping(test_lf, outlier_bounds, split_tag="test")

[CLAMP] train : applied clamping to 76 columns
[CLAMP] test : applied clamping to 76 columns


In [6]:
### STEP 6: High-Dimensional Log1P Transform
# Purpose: Identifies baseline heavy-tailed distributions and seamlessly converts them via logarithmic smoothing.
# Input: Clamped lazy nodes train_lf, test_lf.
# Output: Structurally modified train_lf, test_lf arrays containing scaled _log1p dimensional vectors.
# Expected Side Effects: Lazily aliases and injects math.log1p instructions spanning across the extended feature set.

from src.features import apply_log1p_transform

train_lf = apply_log1p_transform(train_lf, split_tag="train")
test_lf = apply_log1p_transform(test_lf, split_tag="test")

[LOG1P] train : applied log1p to 40 columns -> ['incoming_tx_count', 'outgoing_tx_count', 'net_incoming_tx_count', 'total_gas_paid_eth', 'avg_gas_paid_per_tx_eth', 'risky_tx_count', 'risky_unique_contract_count', 'risky_sum_outgoing_amount_eth', 'outgoing_tx_sum_eth', 'incoming_tx_sum_eth', 'outgoing_tx_avg_eth', 'incoming_tx_avg_eth', 'max_eth_ever', 'min_eth_ever', 'total_balance_eth', 'total_collateral_eth', 'total_collateral_avg_eth', 'total_available_borrows_eth', 'total_available_borrows_avg_eth', 'risk_factor_above_threshold_daily_count', 'borrow_amount_sum_eth', 'borrow_amount_avg_eth', 'borrow_count', 'repay_amount_sum_eth', 'repay_amount_avg_eth', 'repay_count', 'borrow_repay_diff_eth', 'deposit_count', 'deposit_amount_sum_eth', 'withdraw_amount_sum_eth', 'withdraw_deposit_diff_if_positive_eth', 'liquidation_count', 'liquidation_amount_sum_eth', 'unique_borrow_protocol_count', 'unique_lending_protocol_count', 'LTV_Utilization_Quotient', 'Debt_Service_Coverage_Proxy', 'Net_Une

In [7]:
### STEP 7: Storage Streaming Sink & Physical Data Integrity Assertion Gate
# Purpose: Materialize the complete un-truncated feature space securely to persistent disk and run eager assertions.
# Input: Fully integrated train_lf and test_lf execution graphs.
# Output: Exported Parquet stores ensuring zero-null data contracts.
# Expected Side Effects: Forces CPU execution compilation. Executes disk-level writes. Asserts physical shapes eagerly.

from src.features import assert_feature_integrity

PROJECT_ROOT.joinpath("data/processed").mkdir(parents=True, exist_ok=True)

# Stream execution logic natively to disk preserving full dimensions
train_lf.sink_parquet(str(TRAIN_FEATURES))
test_lf.sink_parquet(str(TEST_FEATURES))

# Hard disk-level existence checks
assert TRAIN_FEATURES.exists() and TRAIN_FEATURES.stat().st_size > 0, "Train feature contract missing on disk."
assert TEST_FEATURES.exists() and TEST_FEATURES.stat().st_size > 0, "Test feature contract missing on disk."

# Eager load for absolute structural verification gate
train_final = pl.read_parquet(str(TRAIN_FEATURES))
test_final = pl.read_parquet(str(TEST_FEATURES))

expected_rows_train = pl.scan_parquet(str(TRAIN_RAW_CLEAN)).select(pl.len()).collect().item()
expected_rows_test = pl.scan_parquet(str(TEST_RAW_CLEAN)).select(pl.len()).collect().item()
expected_cols = train_final.shape[1]

assert_feature_integrity(train_final, expected_rows=expected_rows_train, expected_cols=expected_cols, split_tag="train_final")
assert_feature_integrity(test_final, expected_rows=expected_rows_test, expected_cols=expected_cols, split_tag="test_final")

print(f"[VERIFIED] Complete Engineering Manifold: {expected_cols} structural feature vectors successfully committed to storage.")

[ASSERT] train_final rows   : PASS (402754 rows)
[ASSERT] train_final cols   : PASS (123 cols)
[ASSERT] train_final baseline nulls : PASS (0 nulls)
[ASSERT] train_final eng. nulls     : LOG (found 163066 legitimate nulls)
[ASSERT] train_final target : PASS (column exists)
[INTEGRITY] train_final : ALL CHECKS PASSED
[ASSERT] test_final rows   : PASS (40207 rows)
[ASSERT] test_final cols   : PASS (123 cols)
[ASSERT] test_final baseline nulls : PASS (0 nulls)
[ASSERT] test_final eng. nulls     : LOG (found 13304 legitimate nulls)
[ASSERT] test_final target : PASS (column exists)
[INTEGRITY] test_final : ALL CHECKS PASSED
[VERIFIED] Complete Engineering Manifold: 123 structural feature vectors successfully committed to storage.
